# Evaluating Amazon Mistral Lite 7B finetuned for Skeptic Justifications dataset

### Install all dependencies

In [ ]:
# !pip install  accelerate --progress-bar off
# !pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
# !pip install  peft --progress-bar off
# !pip install  bitsandbytes --progress-bar off
# !pip install git+https://github.com/huggingface/transformers
# !pip install  xformers==0.0.21
# !pip install git+https://github.com/huggingface/trl.git
# !pip install deepspeed==0.9.5
# !pip install wandb
# !pip install vllm bert_score rouge nltk

### Loading Required Libraries

Next, we will load the required libraries for fine-tuning a Large Language Model (LLM)

In [11]:
import nltk
import torch
import random
import pandas as pd
import numpy as np
from rouge import Rouge
from bert_score import score
from vllm import LLM, SamplingParams
from nltk.translate.meteor_score import meteor_score

nltk.download('wordnet')
nltk.download('omw-1.4')

rouge = Rouge()

[nltk_data] Downloading package wordnet to /home/ec2-user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ec2-user/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
torch_version = torch.__version__
if torch_version == "2.0.1+cu118":
    print(f"Torch version is satisfied: {torch.__version__}")
else:
    print("Torch version should be 2.0.1+cu118. Please ensure that before going further")

Torch version is satisfied: 2.0.1+cu118


### Loading the test set for Skeptic

In [4]:
df = pd.read_csv("data/test_skeptic_df.csv")

## vLLM Inference Server Engine for increased inference throughput and latency

In [5]:
# downloads finetuned model from huggingface hub
finetuned_model = "skshreyas714/skeptic-justify"

llm = LLM(model=finetuned_model, tensor_parallel_size=1)

INFO 10-28 06:38:48 llm_engine.py:72] Initializing an LLM engine with config: model='skshreyas714/skeptic-justify', tokenizer='skshreyas714/skeptic-justify', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, quantization=None, seed=0)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO 10-28 06:41:31 llm_engine.py:207] # GPU blocks: 1330, # CPU blocks: 2048


In [5]:
# ids = [i for i in range(len(df))]
# idx = random.choice(ids)
# print(idx)

# prompt = df.iloc[idx]["text"]
# justify = df.iloc[idx]["Justification"]

# print(f"Claim Summary: {prompt}")
# print(f"Actual Output: {justify}")

In [6]:
test_prompts = df["text"].tolist()

In [7]:
sampling_params = SamplingParams(temperature=0.1, max_tokens=256, presence_penalty=1.5,
                                 frequency_penalty=1.8, top_p=0.9, top_k=50, best_of=10,
                                 skip_special_tokens=True, use_beam_search=False,
                                 early_stopping=False)

outputs = llm.generate(test_prompts, sampling_params)

predicted = []
for output in outputs:
    generated_text = output.outputs[0].text.strip(" ")
    predicted.append(generated_text)
    # print(f"Generated text: {generated_text!r}")

Processed prompts: 100%|██████████| 11/11 [00:12<00:00,  1.12s/it]


### Evaluation with ROUGE, METEOR, BERT-Score metrics

In [8]:
def compute_metrics(generated, reference):
    rouge_scores = rouge.get_scores(generated, reference)
    meteor = meteor_score([reference.split()], generated.split())
    bert_precision, bert_recall, bert_f1 = score([generated], [reference], lang="en")
    bert_f1 = bert_f1.detach().numpy().tolist()[0]
    return {"rouge_scores": rouge_scores, "meteor": meteor, "bert_score": bert_f1}

In [9]:
r, m, b = [], [], []
for i in range(len(df)):
    metrics = compute_metrics(predicted[i], df.iloc[i]["Justification"])
    rouge_m, meteor_m, bert_m = metrics["rouge_scores"][0]["rouge-l"]["f"], metrics["meteor"], metrics["bert_score"]
    r.append(rouge_m)
    m.append(meteor_m)
    b.append(bert_m)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

In [24]:
rouge_s = np.round(np.mean(r), 2)
metoer_s = np.round(np.mean(m), 2)
bert_s = np.round(np.mean(b), 2)

In [25]:
print(f"Rouge Scores: {round(rouge_s,2)*100}%\n")
print(f"Meteor Scores: {round(metoer_s,2)*100}%\n")
print(f"BERT Scores: {round(bert_s,2)*100}%\n")

Rouge Scores: 55.00000000000001%

Meteor Scores: 48.0%

BERT Scores: 91.0%



### Dumping the generated justifications into test dataframe

In [26]:
df["Generated_Justifications"] = predicted

In [27]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,labels,text,Generated_Justifications
0,YadivauqlLQ,TOP TRADING INDICATORS || MY FAVOURITE || STOC...,Amrev,TOP TRADING INDICATORS || MY FAVOURITE || STOC...,My favorite trading indicator,"""My favorite trading indicator""",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=YadivauqlLQ,No claims made,No claims to analyse,neutral,<|prompter|>You are a Financial Contrarian Wri...,No claims to analyse.
1,Ew2R-WBhp0c,SpiceJet Share price up by 20% as Former IndiG...,5paisa,"According to ources, former IndiGo co-founder ...",सूर्स की माने तो इंडिगो के कोफाउंडर राकीश गंगव...,"According to sources, Indigo's co-founder Rake...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAcIBw...,https://www.youtube.com/watch?v=Ew2R-WBhp0c,The financial influencer claims that Indigo's ...,The claim about Rakesh Gangwal selling a porti...,neutral,<|prompter|>You are a Financial Contrarian Wri...,The claim about Rakesh Gangwal selling a signi...
2,w-neJT7eaGU,Zomato Share Price Surges to 52-Week High: Kno...,5paisa,"On 18-Oct-2023, Zomato share price reached a 5...","Hi everyone, aaj Zomato ke shares ne a 52 week...","""Hi everyone, today Zomato's shares have hit a...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgcICAgIBw...,https://www.youtube.com/watch?v=w-neJT7eaGU,The financial influencer claims that Zomato's ...,The claim seems plausible as strategic partner...,true,<|prompter|>You are a Financial Contrarian Wri...,The claim that Zomato's shares have hit a 52-w...
3,2o8ddlQRxyo,Adani Ka Master Plan,Amrev,Adani Ka Master Plan\n\n.........................,अधानी नहीं किया हूँ He was a diamond trader हू...,"""I am not a diamond trader. He was a diamond t...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBgoKCQgICA...,https://www.youtube.com/watch?v=2o8ddlQRxyo,No claims made,No claims to analyse,neutral,<|prompter|>You are a Financial Contrarian Wri...,No claims to analyse.
4,17Ti-ZEj7zE,Common Mistake People Make || Crypto Mining Se...,Amrev,Common Mistake People Make || Crypto Mining Se...,क्रिप्टो माइनिंग में कमाया भी लेकिन कमा तो लिय...,"Text: ""I earned in crypto mining, but even aft...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=17Ti-ZEj7zE,The financial influencer claims that he earned...,The claim that the influencer earned money thr...,neutral,<|prompter|>You are a Financial Contrarian Wri...,The claim about earning money through cryptocu...


In [28]:
df.to_csv("data/mistral-lite-finetuned-justifications.csv", index=False)